# Hadamard NMR hardware preparation

This notebook generates the representative Hadamard pulse families used for the hardware RF-gain experiment from the frozen nominal and risk-aware neural compilers.

The neural controls are represented in the same Hz-equivalent convention used by the simulator. The notebook preserves the complete three-spin propagation model, the 300-slice × 35 μs pulse duration, the RF-gain sweep, phase-sensitive observables, and the hardware run sheet.

**Hardware-calibration boundary.** The 50 kHz value is the numerical optimization bound, not a universal spectrometer amplitude calibration. The notebook deliberately stops at the Hz-equivalent control representation. The instrument-specific Ankara/SpinQ conversion used for the reported hardware run is preserved separately in `scripts/05_export_spinq_waveforms.py` and `data/hardware/spinq/`; it must not be inferred from the 50 kHz numerical optimization bound.

In [ ]:
# ============================================================
# 1. Imports + GPU
# ============================================================
import os, sys, math, json, shutil, zipfile, hashlib, platform, subprocess
from pathlib import Path
from dataclasses import dataclass, fields
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the full hardware-preparation simulation.")

DEVICE=torch.device("cuda")
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32=True

SEED=20260919
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Python:",sys.version)
print("PyTorch:",torch.__version__)
print("CUDA:",torch.version.cuda)
print("GPU:",torch.cuda.get_device_name(0))

def sha256_file(path,block=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(block),b""):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
# ============================================================
# 2. Load the frozen publication model weights and nominal config
# ============================================================
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

NOMINAL_PATH = REPO_ROOT / "data" / "checkpoints" / "nominal_best_state_dict.pt"
ROBUST_PATH = REPO_ROOT / "data" / "checkpoints" / "robust_final_state_dict.pt"
NOMINAL_CONFIG_PATH = REPO_ROOT / "configs" / "nominal.json"
NOMINAL_SUMMARY_PATH = REPO_ROOT / "results" / "nominal" / "final_validation_summary.json"

for path in [NOMINAL_PATH, ROBUST_PATH, NOMINAL_CONFIG_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required publication input not found: {path}"
        )

nom_obj = torch.load(
    NOMINAL_PATH,
    map_location=DEVICE,
    weights_only=False,
)

with NOMINAL_CONFIG_PATH.open() as handle:
    CONFIG_DICT = json.load(handle)

nominal_reference_summary = None
if NOMINAL_SUMMARY_PATH.exists():
    with NOMINAL_SUMMARY_PATH.open() as handle:
        nominal_reference_summary = json.load(handle)

print("Nominal state dict:", NOMINAL_PATH.relative_to(REPO_ROOT))
print("Nominal SHA256:", sha256_file(NOMINAL_PATH))
print("Robust state dict:", ROBUST_PATH.relative_to(REPO_ROOT))
print("Robust SHA256:", sha256_file(ROBUST_PATH))
print("Nominal config:", NOMINAL_CONFIG_PATH.relative_to(REPO_ROOT))
print("Selected nominal training step: 48600")
if nominal_reference_summary is not None:
    print("Frozen nominal validation statistics:", nominal_reference_summary)

In [ ]:
# ============================================================
# 3. Exact project configuration, network, and 3-spin physics
# ============================================================
@dataclass
class Config:
    v_hz: tuple=(-921.0,40.75,700.0)
    j_hz: tuple=(-64.0,24.4,34.1)
    n_slices:int=300
    dt_s:float=35e-6
    rabi_max_hz:float=50_000.0
    width:int=256
    hidden_layers:int=10
    dropout:float=.25
    batch_size:int=32
    train_steps:int=50_000
    lr:float=5e-4
    weight_decay:float=1e-3
    grad_clip:float=1.0
    smooth_lambda:float=1e-4
    power_lambda:float=1e-6
    val_gates:int=512
    val_batch:int=64
    validate_every:int=100
    print_every:int=20
    seed:int=20260917
    out_dir:str="outputs/nominal"

allowed={f.name for f in fields(Config)}
CFG=Config(**{k:v for k,v in CONFIG_DICT.items() if k in allowed})
CFG.v_hz=tuple(CFG.v_hz)
CFG.j_hz=tuple(CFG.j_hz)

assert CFG.n_slices==300
assert abs(CFG.dt_s-35e-6)<1e-12
assert CFG.hidden_layers==10

print(CFG)
print("Total pulse duration (ms):",CFG.n_slices*CFG.dt_s*1e3)

def kron3(a,b,c):
    return torch.kron(torch.kron(a,b),c)

class NMRSystem:
    def __init__(self,cfg,device):
        ct=torch.complex64
        I=torch.eye(2,dtype=ct,device=device)
        X=torch.tensor([[0,1],[1,0]],dtype=ct,device=device)
        Y=torch.tensor([[0,-1j],[1j,0]],dtype=ct,device=device)
        Z=torch.tensor([[1,0],[0,-1]],dtype=ct,device=device)

        self.I2=I
        self.I4=torch.eye(4,dtype=ct,device=device)
        self.I8=torch.eye(8,dtype=ct,device=device)
        self.X,self.Y,self.Z=X,Y,Z

        self.Xs=[kron3(X,I,I),kron3(I,X,I),kron3(I,I,X)]
        self.Ys=[kron3(Y,I,I),kron3(I,Y,I),kron3(I,I,Y)]
        self.Zs=[kron3(Z,I,I),kron3(I,Z,I),kron3(I,I,Z)]

        self.Hx=sum(self.Xs)
        self.Hy=sum(self.Ys)

        self.pair_ops=[]
        for i,j in [(0,1),(0,2),(1,2)]:
            self.pair_ops.append(
                self.Xs[i]@self.Xs[j]
                +self.Ys[i]@self.Ys[j]
                +self.Zs[i]@self.Zs[j]
            )

        v=torch.tensor(cfg.v_hz,dtype=torch.float32,device=device)[None,:]
        j=torch.tensor(cfg.j_hz,dtype=torch.float32,device=device)[None,:]
        self.H0=self.build_h0(v,j)[0]

    def build_h0(self,v,j):
        H=torch.zeros((len(v),8,8),dtype=torch.complex64,device=v.device)
        for i in range(3):
            H += math.pi*v[:,i,None,None].to(H.dtype)*self.Zs[i]
        for k in range(3):
            H += math.pi*j[:,k,None,None].to(H.dtype)*self.pair_ops[k]
        return H

SYS=NMRSystem(CFG,DEVICE)

class PulseCompiler(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        layers=[]
        d=4
        for _ in range(cfg.hidden_layers):
            layers += [nn.Linear(d,cfg.width),nn.GELU()]
            if cfg.dropout>0:
                layers.append(nn.Dropout(cfg.dropout))
            d=cfg.width
        self.backbone=nn.Sequential(*layers)
        self.head=nn.Linear(d,2*cfg.n_slices)
        self.cfg=cfg

    def forward(self,q):
        raw=self.head(self.backbone(q)).view(-1,self.cfg.n_slices,2)
        r2=(raw*raw).sum(-1,keepdim=True)
        return self.cfg.rabi_max_hz*raw/torch.sqrt(1+r2)

def extract_state(obj):
    if isinstance(obj,dict) and "model" in obj:
        return obj["model"]
    return obj

def unitary_fidelity(target,U):
    ov=(target.conj()*U).sum((-2,-1))
    return ov.abs().square()/64.0

def lift_target(U2):
    return torch.einsum("bij,kl->bikjl",U2,SYS.I4).reshape(-1,8,8)

def propagate(controls,rf_scale=1.0):
    u=controls*float(rf_scale)
    ux,uy=u[...,0],u[...,1]

    H=(
        SYS.H0[None,None,:,:]
        +math.pi*ux[...,None,None].to(torch.complex64)*SYS.Hx
        +math.pi*uy[...,None,None].to(torch.complex64)*SYS.Hy
    )

    Usteps=torch.matrix_exp((-1j*CFG.dt_s)*H)

    U=SYS.I8[None,:,:].expand(len(controls),-1,-1).clone()
    for t in range(CFG.n_slices):
        U=Usteps[:,t]@U
    return U

In [ ]:
# ============================================================
# 4. Load frozen nominal and robust neural compilers
# ============================================================
rob_obj = torch.load(
    ROBUST_PATH,
    map_location=DEVICE,
    weights_only=False,
)

NOMINAL_NN = PulseCompiler(CFG).to(DEVICE)
ROBUST_NN = PulseCompiler(CFG).to(DEVICE)

NOMINAL_NN.load_state_dict(extract_state(nom_obj), strict=True)
ROBUST_NN.load_state_dict(extract_state(rob_obj), strict=True)

for model in [NOMINAL_NN, ROBUST_NN]:
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)

print("Loaded both frozen compilers.")

In [ ]:
# ============================================================
# 5. Construct Hadamard target in SU(2)
# ============================================================
X=SYS.X
Y=SYS.Y
Z=SYS.Z

H_u2=(X+Z)/math.sqrt(2)

# det(H)=-1. Multiplication by i gives an SU(2) representative.
# Global phase is irrelevant for the project fidelity.
H_su2=1j*H_u2

def u2_to_q(U):
    q0=(torch.trace(U).real/2)
    qx=(0.5j*torch.trace(X@U)).real
    qy=(0.5j*torch.trace(Y@U)).real
    qz=(0.5j*torch.trace(Z@U)).real
    q=torch.stack([q0,qx,qy,qz]).float()
    return q/q.norm()

Q_H=u2_to_q(H_su2)[None,:]
TARGET_H=lift_target(H_su2[None,:,:])

print("Hadamard quaternion:",Q_H[0].cpu().numpy())
print("Quaternion norm:",float(Q_H.norm()))
print("det(H_su2):",torch.linalg.det(H_su2))

In [ ]:
# ============================================================
# 6. Compile the two base Hadamard waveforms
# ============================================================
with torch.inference_mode():
    UCTRL_NOM=NOMINAL_NN(Q_H)
    UCTRL_ROB=ROBUST_NN(Q_H)

    U_NOM_0=propagate(UCTRL_NOM,1.0)
    U_ROB_0=propagate(UCTRL_ROB,1.0)

    F_NOM_0=float(unitary_fidelity(TARGET_H,U_NOM_0).item())
    F_ROB_0=float(unitary_fidelity(TARGET_H,U_ROB_0).item())

print("Nominal-NN Hadamard fidelity @ 0% RF error:",F_NOM_0)
print("Robust-NN Hadamard fidelity @ 0% RF error:",F_ROB_0)

for label,u in [("Nominal NN",UCTRL_NOM[0]),("Robust NN",UCTRL_ROB[0])]:
    amp=torch.linalg.vector_norm(u,dim=-1)
    print(
        label,
        "| peak Hz =",float(amp.max()),
        "| RMS Hz =",float(torch.sqrt((amp**2).mean()))
    )

In [ ]:
# ============================================================
# 7. Exact RF sweep + simulated hardware observables
# ============================================================
RF_ERRORS_PERCENT=np.array(
    [-10.0,-7.5,-5.0,-2.5,0.0,2.5,5.0,7.5,10.0],
    dtype=np.float64
)

RHO0=SYS.Zs[0]
OBS_X=SYS.Xs[0]
OBS_Y=SYS.Ys[0]
OBS_Z=SYS.Zs[0]

NORM_X=torch.trace(OBS_X.conj().T@OBS_X).real
NORM_Y=torch.trace(OBS_Y.conj().T@OBS_Y).real
NORM_Z=torch.trace(OBS_Z.conj().T@OBS_Z).real

def transfer_observables(U):
    rho=U@RHO0@U.conj().T

    mx=(torch.trace(OBS_X@rho).real/NORM_X).item()
    my=(torch.trace(OBS_Y@rho).real/NORM_Y).item()
    mz=(torch.trace(OBS_Z@rho).real/NORM_Z).item()

    mperp=math.sqrt(mx*mx+my*my)
    phase=math.atan2(my,mx)

    return dict(
        Mx=float(mx),
        My=float(my),
        Mz=float(mz),
        Mperp=float(mperp),
        phase_rad=float(phase),
        phase_deg=float(math.degrees(phase)),
    )

rows=[]
unitaries={}

for method,controls in [
    ("Nominal NN",UCTRL_NOM),
    ("Robust NN",UCTRL_ROB),
]:
    for err in RF_ERRORS_PERCENT:
        scale=1.0+err/100.0

        with torch.inference_mode():
            U=propagate(controls,scale)[0]

        F=float(unitary_fidelity(TARGET_H,U[None,:,:]).item())
        obs=transfer_observables(U)

        rows.append({
            "method":method,
            "rf_error_percent":float(err),
            "rf_scale":float(scale),
            "unitary_fidelity":F,
            **obs,
        })

        key=f"{method.replace(' ','_')}_{err:+05.1f}pct"
        unitaries[key]=U.detach().cpu().numpy()

RESULTS=pd.DataFrame(rows)
display(RESULTS)

print("\nRepresentative -5%, 0%, +5% conditions:")
display(
    RESULTS[
        RESULTS.rf_error_percent.isin([-5.0,0.0,5.0])
    ]
)

## How to read the predicted hardware observables

For this representative experiment the ideal transfer is

\[
Z_1\rightarrow X_1.
\]

Therefore:

- `Mx = +1` is ideal phase-sensitive transfer.
- `Mperp = 1` is ideal transverse magnitude.
- `phase_deg = 0°` is ideal transverse phase.
- `Mz = 0` means no residual longitudinal target-spin component.

The hardware acquisition should retain the **complex FID/spectrum**, not only absolute peak height. A large `Mperp` with the wrong phase is not the desired \(+X\) output.

The rigorous gate-level quantity remains the simulated `unitary_fidelity`. The experimentally measured spectrum is compared with `Mx`, `My`, `Mperp`, and `phase_deg`.

The exported waveforms are deliberately left in the model's Hz-equivalent convention. The downstream Ankara/SpinQ conversion used for the reported experiment is implemented in `scripts/05_export_spinq_waveforms.py`; the 50 kHz numerical optimization bound is not used as the device full-scale calibration.

In [ ]:
# ============================================================
# 8. Create complete experiment export tree + waveform files
# ============================================================
OUT = REPO_ROOT / "outputs" / "hardware_preparation"
shutil.rmtree(OUT, ignore_errors=True)

RAW=OUT/"raw"
PULSES=OUT/"pulses"
TABLES=OUT/"tables"
FIGS=OUT/"figures"
META=OUT/"metadata"

for d in [OUT,RAW,PULSES,TABLES,FIGS,META]:
    d.mkdir(parents=True,exist_ok=True)

with open(META / "nominal_config.json", "w") as f:
    json.dump(CONFIG_DICT, f, indent=2, default=str)

shutil.copy2(NOMINAL_PATH, META / "nominal_best_state_dict.pt")
shutil.copy2(ROBUST_PATH, META / "robust_final_state_dict.pt")

np.save(RAW/"hadamard_u2.npy",H_u2.detach().cpu().numpy())
np.save(RAW/"hadamard_su2.npy",H_su2.detach().cpu().numpy())
np.save(RAW/"hadamard_target_3spin.npy",TARGET_H[0].detach().cpu().numpy())
np.save(RAW/"hadamard_quaternion.npy",Q_H[0].detach().cpu().numpy())
np.save(RAW/"H0_full_XX_YY_ZZ.npy",SYS.H0.detach().cpu().numpy())
np.save(RAW/"initial_deviation_Z1.npy",RHO0.detach().cpu().numpy())
np.save(RAW/"observable_X1.npy",OBS_X.detach().cpu().numpy())
np.save(RAW/"observable_Y1.npy",OBS_Y.detach().cpu().numpy())
np.save(RAW/"observable_Z1.npy",OBS_Z.detach().cpu().numpy())

for key,U in unitaries.items():
    np.save(RAW/f"unitary_{key}.npy",U)

RESULTS.to_csv(TABLES/"simulated_hadamard_rf_sweep.csv",index=False)

base_controls={
    "nominal_nn":UCTRL_NOM[0].detach().cpu().numpy(),
    "robust_nn":UCTRL_ROB[0].detach().cpu().numpy(),
}

def pulse_tag(err):
    return (
        f"rf_{err:+05.1f}pct"
        .replace("+","plus")
        .replace("-","minus")
        .replace(".","p")
    )

for method_key,u0 in base_controls.items():
    method_dir=PULSES/method_key
    method_dir.mkdir(exist_ok=True)

    np.save(method_dir/"base_controls_ux_uy_hz.npy",u0)

    for err in RF_ERRORS_PERCENT:
        scale=1+err/100.0
        u=u0*scale

        ux=u[:,0]
        uy=u[:,1]
        amp=np.sqrt(ux**2+uy**2)
        phase=np.arctan2(uy,ux)

        tag=pulse_tag(err)

        df=pd.DataFrame({
            "slice":np.arange(CFG.n_slices),
            "time_us":np.arange(CFG.n_slices)*CFG.dt_s*1e6,
            "dwell_us":np.full(CFG.n_slices,CFG.dt_s*1e6),
            "ux_hz":ux,
            "uy_hz":uy,
            "amplitude_hz":amp,
            "amplitude_fraction_of_numeric_cap":amp/CFG.rabi_max_hz,
            "phase_rad":phase,
            "phase_deg":np.degrees(phase),
            "rf_error_percent":np.full(CFG.n_slices,err),
            "rf_scale":np.full(CFG.n_slices,scale),
        })

        df.to_csv(method_dir/f"{tag}.csv",index=False)

        np.savez(
            method_dir/f"{tag}.npz",
            ux_hz=ux,
            uy_hz=uy,
            amplitude_hz=amp,
            amplitude_fraction_of_numeric_cap=amp/CFG.rabi_max_hz,
            phase_rad=phase,
            phase_deg=np.degrees(phase),
            dwell_s=CFG.dt_s,
            rf_error_percent=err,
            rf_scale=scale,
        )

print("Export root:",OUT)

In [ ]:
# ============================================================
# 9. Hardware run sheet
# ============================================================
run_rows=[]
condition_id=0

for err in RF_ERRORS_PERCENT:
    for method in ["Nominal NN","Robust NN"]:
        condition_id+=1

        row=RESULTS[
            (RESULTS.method==method)
            &(RESULTS.rf_error_percent==err)
        ].iloc[0]

        method_key="nominal_nn" if method=="Nominal NN" else "robust_nn"
        tag=pulse_tag(err)

        run_rows.append({
            "condition_id":condition_id,
            "method":method,
            "rf_error_percent":err,
            "rf_scale":1+err/100,
            "waveform_csv":f"pulses/{method_key}/{tag}.csv",
            "waveform_npz":f"pulses/{method_key}/{tag}.npz",
            "simulated_unitary_fidelity":row.unitary_fidelity,
            "simulated_Mx":row.Mx,
            "simulated_My":row.My,
            "simulated_Mz":row.Mz,
            "simulated_Mperp":row.Mperp,
            "simulated_phase_deg":row.phase_deg,
            "recommended_repeats":3,
        })

RUN_SHEET=pd.DataFrame(run_rows)
RUN_SHEET.to_csv(TABLES/"hardware_run_sheet.csv",index=False)
display(RUN_SHEET)

In [ ]:
# ============================================================
# 10. Simulation prediction plots
# ============================================================
def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGS/f"{name}.png",dpi=600,bbox_inches="tight")
    plt.savefig(FIGS/f"{name}.pdf",bbox_inches="tight")
    plt.show()
    plt.close()

plt.figure(figsize=(7.8,5))
for method in ["Nominal NN","Robust NN"]:
    d=RESULTS[RESULTS.method==method]
    plt.plot(d.rf_error_percent,d.unitary_fidelity,marker="o",label=method)
plt.xlabel("Global RF amplitude error (%)")
plt.ylabel("Three-spin unitary fidelity")
plt.ylim(0,1.01)
plt.grid(alpha=.2)
plt.legend()
savefig("simulated_unitary_fidelity_vs_rf_error")

plt.figure(figsize=(7.8,5))
for method in ["Nominal NN","Robust NN"]:
    d=RESULTS[RESULTS.method==method]
    plt.plot(d.rf_error_percent,d.Mx,marker="o",label=method)
plt.axhline(1.0,ls="--",lw=1)
plt.xlabel("Global RF amplitude error (%)")
plt.ylabel("Normalized target +X response, Mx")
plt.grid(alpha=.2)
plt.legend()
savefig("simulated_Mx_vs_rf_error")

plt.figure(figsize=(7.8,5))
for method in ["Nominal NN","Robust NN"]:
    d=RESULTS[RESULTS.method==method]
    plt.plot(d.rf_error_percent,d.Mperp,marker="o",label=method)
plt.axhline(1.0,ls="--",lw=1)
plt.xlabel("Global RF amplitude error (%)")
plt.ylabel("Normalized transverse magnitude")
plt.grid(alpha=.2)
plt.legend()
savefig("simulated_Mperp_vs_rf_error")

plt.figure(figsize=(7.8,5))
for method in ["Nominal NN","Robust NN"]:
    d=RESULTS[RESULTS.method==method]
    plt.plot(d.rf_error_percent,d.phase_deg,marker="o",label=method)
plt.axhline(0.0,ls="--",lw=1)
plt.xlabel("Global RF amplitude error (%)")
plt.ylabel("Transverse phase (deg)")
plt.grid(alpha=.2)
plt.legend()
savefig("simulated_phase_vs_rf_error")

plt.figure(figsize=(9,5))
for label,u in [
    ("Nominal NN",base_controls["nominal_nn"]),
    ("Robust NN",base_controls["robust_nn"])
]:
    amp=np.sqrt((u*u).sum(1))
    plt.plot(np.arange(CFG.n_slices)*CFG.dt_s*1e3,amp,label=label)
plt.xlabel("Time (ms)")
plt.ylabel("RF amplitude (Hz)")
plt.grid(alpha=.2)
plt.legend()
savefig("hadamard_base_pulse_amplitude")

In [ ]:
# ============================================================
# 11. Metadata, README, manifest, checksums
# ============================================================
experiment_spec={
    "experiment":"Representative Hadamard NMR robustness demonstration",
    "gate":"Hadamard",
    "initial_deviation_operator":"Z1",
    "ideal_output_operator":"X1",
    "methods":["Nominal NN","Robust NN"],
    "rf_error_percent":RF_ERRORS_PERCENT.tolist(),
    "recommended_repeats_per_condition":3,
    "n_conditions":int(len(RF_ERRORS_PERCENT)*2),
    "n_total_acquisitions_if_3_repeats":int(len(RF_ERRORS_PERCENT)*2*3),
    "pulse":{
        "n_slices":CFG.n_slices,
        "dwell_s":CFG.dt_s,
        "dwell_us":CFG.dt_s*1e6,
        "total_duration_s":CFG.n_slices*CFG.dt_s,
        "total_duration_ms":CFG.n_slices*CFG.dt_s*1e3,
        "numeric_rf_cap_hz":CFG.rabi_max_hz,
    },
    "physics":{
        "v_hz":list(CFG.v_hz),
        "j_hz":list(CFG.j_hz),
        "interaction":"XX+YY+ZZ using the current project Hamiltonian convention",
    },
    "important_note":
        "The 50 kHz RF cap is the numerical training cap, not an experimentally calibrated instrument maximum. "
        "The recorded Ankara/SpinQ device conversion is implemented separately in scripts/05_export_spinq_waveforms.py.",
}

with open(META/"experiment_spec.json","w") as f:
    json.dump(experiment_spec,f,indent=2)

env={
    "python":sys.version,
    "platform":platform.platform(),
    "torch":torch.__version__,
    "numpy":np.__version__,
    "pandas":pd.__version__,
    "cuda":torch.version.cuda,
    "gpu":torch.cuda.get_device_name(0),
    "nominal_checkpoint":str(NOMINAL_PATH.relative_to(REPO_ROOT)),
    "robust_checkpoint":str(ROBUST_PATH.relative_to(REPO_ROOT)),
    "nominal_checkpoint_sha256":sha256_file(NOMINAL_PATH),
    "robust_checkpoint_sha256":sha256_file(ROBUST_PATH),
}

with open(META/"environment_and_provenance.json","w") as f:
    json.dump(env,f,indent=2)

readme=f'''
HADAMARD HARDWARE NMR ROBUSTNESS EXPERIMENT
=====================================================

TARGET
------
Hadamard gate on spin 1:
    Z1 -> X1

METHODS
-------
1. Nominal neural compiler
2. Robust neural compiler

CONTROLLED PERTURBATION
-----------------------
Global RF amplitude scaling:
    -10, -7.5, -5, -2.5, 0, +2.5, +5, +7.5, +10 %

PULSE
-----
Slices:           {CFG.n_slices}
Dwell:            {CFG.dt_s*1e6:.6f} us
Total duration:   {CFG.n_slices*CFG.dt_s*1e3:.6f} ms
Numerical RF cap: {CFG.rabi_max_hz:.6f} Hz

IMPORTANT HARDWARE NOTE
-----------------------
The pulse files are expressed in model Hz plus amplitude/phase.
The 50 kHz numerical cap is NOT the calibrated spectrometer maximum.
The Hz-equivalent controls are converted to the device format by
scripts/05_export_spinq_waveforms.py using the recorded Ankara/SpinQ convention.
The 50 kHz numerical cap is not used as the device full-scale calibration.

WHAT TO ACQUIRE
---------------
Save the full complex FID for every condition.
Use consistent receiver phase and spectral processing.

Primary measured quantities:
    Mx      desired phase-sensitive +X response
    My      quadrature error
    Mperp   transverse magnitude
    phase   transverse phase

Do not call Mperp alone "gate fidelity".
The rigorous gate-level reference is the simulated unitary fidelity.

PRIMARY HARDWARE HYPOTHESIS
---------------------------
Nominal NN should show a sharper optimum near zero RF error.
Robust NN should retain substantially more target response away from zero,
especially around +/-5 percent and beyond.
'''

(OUT/"README_EXPERIMENT.txt").write_text(readme)

manifest=[
    str(p.relative_to(OUT))
    for p in sorted(OUT.rglob("*"))
    if p.is_file()
]

with open(OUT/"MANIFEST.txt","w") as f:
    f.write("\n".join(manifest))

checks=[]
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.name!="SHA256SUMS.txt":
        checks.append(f"{sha256_file(p)}  {p.relative_to(OUT)}")

with open(OUT/"SHA256SUMS.txt","w") as f:
    f.write("\n".join(checks))

print("Prepared",len(manifest),"files.")

In [ ]:
# ============================================================
# 12. Final summary + ZIP + browser download
# ============================================================
print("="*76)
print("HADAMARD RF-SWEEP HARDWARE PREPARATION COMPLETE")
print("="*76)

print("\nSIMULATED RESULTS")
display(RESULTS)

print("\nHARDWARE RUN SHEET")
display(RUN_SHEET)

ZIP_PATH=shutil.make_archive(
    str(OUT),
    "zip",
    root_dir=OUT.parent,
    base_dir=OUT.name
)

print("\nZIP:",ZIP_PATH)
print("ZIP size:",f"{os.path.getsize(ZIP_PATH)/1024**2:.2f} MB")

print('Archive written to:', ZIP_PATH)